In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [2]:
v1.dot(dv)

np.float64(0.3233238326017993)

In [3]:
v2.dot(dv)

np.float64(0.019730417971579928)

In [4]:
q1 = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)

In [5]:
v1

array([-2.05820368e-02, -1.40459110e-02,  3.02994368e-02, -5.40378223e-02,
        7.18781390e-02, -2.79537319e-02, -5.03093720e-02, -1.27217001e-02,
        4.08207972e-02, -2.60037183e-02,  3.05458390e-02,  4.21485157e-02,
        8.09861810e-02, -6.93957347e-02, -1.30190525e-01, -6.39247898e-02,
        4.81059739e-02,  1.60095563e-02, -5.22432375e-02, -7.13635090e-02,
       -3.83860331e-03,  2.48125820e-02,  4.40211522e-02, -3.45579342e-02,
        1.52686273e-02,  7.61531501e-03,  5.38177740e-02,  1.18252761e-02,
        1.32433854e-02,  2.89461731e-02,  6.64911339e-03,  7.04788251e-02,
        6.19290516e-02,  2.11051650e-02, -7.33482725e-02,  2.84129310e-02,
       -3.72108408e-02,  6.22799424e-02, -4.86973127e-02,  4.49664033e-02,
       -2.59480721e-02,  2.04324107e-02,  1.79650516e-02,  1.02705477e-02,
        2.74898638e-03,  2.84324658e-02, -3.30318144e-02,  6.70969300e-02,
       -1.56520359e-02, -8.51907947e-02, -1.24307168e-01,  4.32504168e-02,
       -5.64432917e-02,  

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [7]:
# Assuming your list is called 'documents'
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

In [8]:
index = -1  # Default value if not found

for i, doc in enumerate(documents):
    if doc.get('filename') == target_filename:
        index = i
        break  # Exit early once found

print(index)

22


In [9]:
documents[22]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [10]:
v2=embed.encode(documents[22]['content'])

In [11]:
v1.dot(v2)

np.float64(0.361070280302606)

In [12]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [58]:
chunks[0:4]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [49]:
chunks[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

In [44]:
chunks[1]['content']

'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometimes produce confident-soundin

In [62]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch_chunks = chunks[i:i + batch_size]
    batch_chunks_content=[sub_chunks['content'] for sub_chunks in batch_chunks]
    batch_vectors = embed.encode_batch(batch_chunks_content)
    vectors.extend(batch_vectors)

len(vectors)

import numpy as np
X = np.array(vectors)

  0%|          | 0/6 [00:00<?, ?it/s]

In [63]:
X.shape

(295, 384)

In [64]:
scores = X.dot(v1)

In [65]:
scores

array([ 3.15187667e-01,  2.01479590e-01,  5.90558833e-02, -7.67662092e-02,
        1.18452457e-01, -1.41697008e-01, -2.81406239e-02, -4.65670219e-02,
       -2.06994421e-02, -6.07744072e-02,  2.13273697e-01,  8.87601283e-02,
       -1.97269148e-02,  3.11629945e-01,  5.51034649e-01,  2.04008064e-01,
        2.12515834e-01,  1.93649166e-01,  2.51961211e-01,  1.31078657e-01,
        2.59120526e-01,  7.63816029e-02,  9.59192920e-02,  9.81469839e-03,
       -3.59106955e-02,  1.01211556e-02,  1.10786957e-01, -9.90259327e-02,
       -3.71170699e-02,  7.59057318e-02, -3.35340264e-02,  8.86838363e-03,
        1.02636359e-01,  6.89614769e-02,  1.29408804e-01,  2.57709174e-01,
        3.23680535e-01,  1.06350088e-01,  5.61891403e-02,  2.34017413e-01,
        1.97954389e-01,  9.64296226e-02,  1.93709947e-01,  2.16719278e-01,
        3.48340450e-01,  5.10906387e-02,  2.05212841e-01,  1.05416126e-01,
       -3.25432324e-02,  4.94665571e-02,  2.38574833e-01, -3.44206606e-02,
        1.82165502e-01,  

In [66]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float64(0.648901732433228))

In [67]:
chunks[94]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [72]:
documents[0:2]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [74]:
from minsearch import VectorSearch

vindex = VectorSearch(
    keyword_fields=["filename","content"])
vindex.fit(X, chunks)

In [78]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [79]:
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

In [81]:
from minsearch import Index
index_chunk = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunk.fit(chunks)

In [84]:
question = "How do I store vectors in PostgreSQL?"

search_results = index_chunk.search(
    question,
    num_results=5
)

filenames = [result['filename'] for result in search_results]
filenames

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [85]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
filename_result = [result['filename'] for result in results]
filename_result

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']